# BigQuery exploration in the sage-baker project

Two ways to read BQ data: the `%%bigquery` cell magic (ergonomic) and the
direct `google.cloud.bigquery` Python client (more control).

The kernel needs to be **Python (sage-baker)** — pick it from the kernel
switcher in the top right if it isn't already selected.

Prereqs:
- `make install-jupyter` once (gets you `bigquery-magics`)
- `.env` with `GOOGLE_APPLICATION_CREDENTIALS` and `GOOGLE_CLOUD_PROJECT`
- `make bq-upload-sonar` once if you want to query the sonar table
- Launch via `make jupyter` so `.env` is exported to the kernel

In [1]:
# project setup — chdir to project root so relative paths work, src/ on sys.path
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'src')

%load_ext autoreload
%autoreload 2
%load_ext bigquery_magics

PROJECT = os.environ.get('GOOGLE_CLOUD_PROJECT')
print('GOOGLE_CLOUD_PROJECT =', PROJECT)
print('cwd =', os.getcwd())

GOOGLE_CLOUD_PROJECT = gen-lang-client-0749458129
cwd = /home/dan/sandbox/dnewcome/sage-baker


## Pattern 1: `%%bigquery` cell magic

Result lands in the named variable as a `pandas.DataFrame`.

In [2]:
%%bigquery sonar_df --project $PROJECT
SELECT * FROM `gen-lang-client-0749458129.sage_baker.sonar` LIMIT 10

Query is running:   0%|          |

Downloading:   0%|          |

In [3]:
sonar_df.shape, sonar_df['target'].value_counts().to_dict()

((10, 61), {np.int64(0): 10})

## Pattern 2: direct `google.cloud.bigquery` client

More control — useful for parameterized queries, loading larger
datasets via the Storage API, or building dynamic SQL.

In [4]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

df = client.query(f"""
    SELECT *
    FROM `{PROJECT}.sage_baker.sonar`
""").to_dataframe()

print(f'{len(df)} rows × {len(df.columns)} cols')
df.describe().T.head(8)

/home/dan/sandbox/dnewcome/sage-baker/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


208 rows × 61 cols


,count,mean,std,min,25%,50%,75%,max
f0,208.0,0.029164,0.022991,0.0015,0.01335,0.0228,0.03555,0.1371
f1,208.0,0.038437,0.03296,0.0006,0.01645,0.0308,0.04795,0.2339
f2,208.0,0.043832,0.038428,0.0015,0.01895,0.0343,0.05795,0.3059
f3,208.0,0.053892,0.046528,0.0058,0.024375,0.04405,0.0645,0.4264
f4,208.0,0.075202,0.055552,0.0067,0.03805,0.0625,0.100275,0.401
f5,208.0,0.10457,0.059105,0.0102,0.067025,0.09215,0.134125,0.3823
f6,208.0,0.121747,0.061788,0.0033,0.0809,0.10695,0.154,0.3729
f7,208.0,0.134799,0.085152,0.0055,0.080425,0.1121,0.1696,0.459


## Train on it

The trainers in `src/` accept any directory containing a CSV or parquet.
We can write the BQ result to a parquet and call the trainer directly.

In [7]:
import bundle, train, importlib
importlib.reload(train)

df.to_parquet('data/training.parquet', index=False)
sys.argv = ['train.py', '--train', 'data', '--model-dir', 'model_nb']
train.main()

loaded data/training.parquet: 208 rows, 61 columns
validation_accuracy=0.7857


In [8]:
# inspect the bundle metadata — including the lineage if data/lineage.json existed
import json
json.loads(open('model_nb/metadata.json').read())

{'saved_at': '2026-05-07T10:19:22.680401+00:00',
 'python': '3.12.9',
 'git_sha': '544178ce8f94f66b701c367da0f5d1e45748db47',
 'validation_accuracy': 0.7857142857142857,
 'n_train': 166,
 'n_test': 42,
 'dataset_file': 'training.parquet',
 'data_lineage': {'source': 'bigquery',
  'project': 'gen-lang-client-0749458129',
  'query': 'SELECT * FROM `gen-lang-client-0749458129.sage_baker.sonar`',
  'snapshot_timestamp': '2026-05-07T10:10:48.904715+00:00',
  'dataset_sha256': 'd1c8a42f2fa6c68c095d5c15639f92fcdadb68303ac5eb1e9a1ba2ef7cfb83f0',
  'dataset_n_rows': 208,
  'dataset_n_cols': 61,
  'materialized_at': '2026-05-07T10:10:51.580904+00:00'}}

## Inference test

In [9]:
model = train.model_fn('model_nb')
X = df.drop(columns=['target']).head(5)
y = df['target'].head(5).tolist()
preds = model.predict(X).tolist()
print('actual:   ', y)
print('predicted:', preds)

actual:    [0, 0, 0, 0, 0]
predicted: [0.0, 0.0, 0.0, 0.0, 0.0]
